# Subsection **2.1.2 Bloch sphere representation**

In [ ]:
!pip install -q git+https://github.com/2forts/qcirclab_repo.git
!pip install -q matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

from qcirclab import Circuit
from qcirclab import gates as qg

In [ ]:
def circuit_unitary(qc: Circuit) -> np.ndarray:
    """Compute the full unitary matrix of a measurement-free circuit."""
    n = qc.n_qubits
    U = np.zeros((2**n, 2**n), dtype=complex)
    for j in range(2**n):
        basis = np.zeros(2**n, dtype=complex)
        basis[j] = 1.0
        tmp = qc.copy().set_statevector(basis)
        U[:, j] = tmp.statevector()
    return U

def equal_up_to_global_phase(U: np.ndarray, V: np.ndarray, atol: float = 1e-9) -> bool:
    u = U.reshape(-1)
    v = V.reshape(-1)
    idx = None
    for k in range(len(v)):
        if abs(v[k]) > atol and abs(u[k]) > atol:
            idx = k
            break
    if idx is None:
        return np.allclose(U, V, atol=atol)
    phase = u[idx] / v[idx]
    return np.allclose(U, phase * V, atol=atol)

def bloch_vector(state: np.ndarray):
    state = np.asarray(state, dtype=complex).reshape(-1)
    if state.shape != (2,):
        raise ValueError("Bloch vector is only defined here for 1-qubit pure states")
    a, b = state
    x = 2 * np.real(np.conj(a) * b)
    y = 2 * np.imag(np.conj(b) * a)
    z = np.abs(a)**2 - np.abs(b)**2
    return float(x), float(y), float(z)

def plot_bloch_state(state: np.ndarray, title: str = "Bloch sphere"):
    x, y, z = bloch_vector(state)
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection="3d")

    u = np.linspace(0, 2*np.pi, 60)
    v = np.linspace(0, np.pi, 30)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, rstride=3, cstride=3, linewidth=0.5, alpha=0.25)

    ax.quiver(0, 0, 0, 1.1, 0, 0, arrow_length_ratio=0.08)
    ax.quiver(0, 0, 0, 0, 1.1, 0, arrow_length_ratio=0.08)
    ax.quiver(0, 0, 0, 0, 0, 1.1, arrow_length_ratio=0.08)
    ax.text(1.18, 0, 0, "x")
    ax.text(0, 1.18, 0, "y")
    ax.text(0, 0, 1.18, "z")

    ax.quiver(0, 0, 0, x, y, z, arrow_length_ratio=0.12, linewidth=2)
    ax.text(0, 0, 1.05, r"$|0\rangle$")
    ax.text(0, 0, -1.15, r"$|1\rangle$")
    ax.set_title(title)
    ax.set_xlim([-1.2, 1.2]); ax.set_ylim([-1.2, 1.2]); ax.set_zlim([-1.2, 1.2])
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    plt.show()

In [ ]:
qc = Circuit(1)
qc.h(0)

state = qc.statevector()
print("Statevector:", state)
plot_bloch_state(state, title="Bloch sphere of H|0>")

# Subsection **2.2.1 General form of 1-qubit unitaries**

In [ ]:
from numpy import pi

phi, theta, lam = pi/3, pi/4, -pi/2

qc = Circuit(1)
qc.rz(phi, 0)
qc.ry(theta, 0)
qc.rz(lam, 0)

print(qc.draw())
print("\nUnitary:")
print(circuit_unitary(qc))

# Subsection **2.2.3 Parametrized rotation gates**

In [ ]:
qc = Circuit(1)
qc.rx(pi/3, 0)
qc.ry(pi/4, 0)
qc.rz(-pi/2, 0)
qc.u(pi/3, pi/2, pi/4, 0)  # general U(theta, phi, lambda)

print(qc.draw())

# Subsection **2.2.4 Gate composition and commutation properties**

In [ ]:
qc1 = Circuit(1)
qc1.rx(pi/3, 0)
qc1.rz(pi/4, 0)
U1 = circuit_unitary(qc1)

qc2 = Circuit(1)
qc2.rz(pi/4, 0)
qc2.rx(pi/3, 0)
U2 = circuit_unitary(qc2)

print(equal_up_to_global_phase(U1, U2))  # False in general

# Subsection **2.3.1 Controlled gates**

In [ ]:
qc = Circuit(2)
qc.cx(0, 1)   # CNOT with qubit 0 as control, 1 as target
qc.cz(0, 1)   # Controlled-Z

print(qc.draw())

# Subsection **2.3.2 SWAP and other two-qubit gates**

In [ ]:
qc = Circuit(2)
qc.swap(0, 1)

print(qc.draw())

# Subsection **2.3.3 Entanglement generation with CNOT and H**

In [ ]:
qc = Circuit(2)
qc.h(0)
qc.cx(0, 1)

state = qc.statevector()
print(state)

# Subsection **2.4.2 Standard universal sets**

In [ ]:
# Example circuit in a logical gate set
qc_logical = Circuit(2)
qc_logical.h(0)
qc_logical.t(0)
qc_logical.cx(0, 1)

# Manual rewrite into the basis {rz, rx, cx}
qc_basis = Circuit(2)
qc_basis.rz(np.pi/2, 0).rx(np.pi/2, 0).rz(np.pi/2, 0)  # H up to global phase
qc_basis.rz(np.pi/4, 0)                                 # T up to global phase
qc_basis.cx(0, 1)

print("Logical circuit:")
print(qc_logical.draw())
print("\nRewritten in {rz, rx, cx}:")
print(qc_basis.draw())
print("\nEquivalent up to global phase:",
      equal_up_to_global_phase(circuit_unitary(qc_logical), circuit_unitary(qc_basis)))

# Subsection *2.4.3 Approximation of arbitrary unitaries*

In [ ]:
target = qg.rz(0.3 * np.pi)

alphabet = {
    "H": qg.H,
    "S": qg.S,
    "T": qg.T,
}
best_seq = None
best_err = np.inf
best_U = None

for length in range(1, 7):
    for seq in product(alphabet.keys(), repeat=length):
        U = np.eye(2, dtype=complex)
        for gate_name in seq:
            U = alphabet[gate_name] @ U
        # compare up to global phase
        flat_t = target.reshape(-1)
        flat_u = U.reshape(-1)
        idx = next(i for i in range(len(flat_t)) if abs(flat_t[i]) > 1e-12)
        phase = flat_u[idx] / flat_t[idx]
        err = np.linalg.norm(U - phase * target)
        if err < best_err:
            best_err = err
            best_seq = seq
            best_U = U

print("Target matrix Rz(0.3π):")
print(target)
print("\nBest short H/S/T sequence found:", best_seq)
print("Approximation error:", best_err)
print("\nApproximate unitary:")
print(best_U)

# Subsection **2.4.4 Native vs logical gate sets**

In [ ]:
# Logical description
qc_logical = Circuit(2)
qc_logical.h(0)
qc_logical.t(0)
qc_logical.cx(0, 1)

# Native description in the basis {rx, rz, cx}
qc_native = Circuit(2)
qc_native.rz(np.pi/2, 0).rx(np.pi/2, 0).rz(np.pi/2, 0)
qc_native.rz(np.pi/4, 0)
qc_native.cx(0, 1)

print("Logical circuit:")
print(qc_logical.draw())
print("\nNative-basis circuit:")
print(qc_native.draw())
print("\nEquivalent up to global phase:",
      equal_up_to_global_phase(circuit_unitary(qc_logical), circuit_unitary(qc_native)))

# Subsection **2.5.1 Matrix factorization and Euler angles for 1-qubit gates**

In [ ]:
from numpy import angle, arccos, sqrt

# Example SU(2) matrix (unitary with det=1)
V = np.array([[np.exp(-1j*(0.6))/sqrt(2), -np.exp(-1j*(0.2))/sqrt(2)],
              [np.exp( 1j*(0.2))/sqrt(2),  np.exp( 1j*(0.6))/sqrt(2)]], dtype=complex)

# Extract ZYZ Euler angles from V up to a global phase
a, b, c, d = V[0,0], V[0,1], V[1,0], V[1,1]
theta = 2 * arccos(np.clip(np.abs(a), 0.0, 1.0))

if np.isclose(np.sin(theta/2), 0.0):
    phi = 0.0
    lam = angle(d) - angle(a)
else:
    phi = angle(c) - angle(a)
    lam = angle(-b) - angle(a)

qc = Circuit(1)
qc.u(theta, phi, lam, 0)
U_rec = circuit_unitary(qc)

print("Recovered angles:")
print("theta =", theta)
print("phi   =", phi)
print("lambda=", lam)
print("\nEquivalent up to global phase:", equal_up_to_global_phase(V, U_rec))

# Subsection **2.5.2 Decomposing controlled operations into elementary gates**

In [ ]:
# Controlled-Z can be decomposed as (I ⊗ H) CNOT (I ⊗ H)

qc_controlled = Circuit(2)
qc_controlled.unitary(qg.Z, [1], name="z", controls=[0])

qc_decomposed = Circuit(2)
qc_decomposed.h(1)
qc_decomposed.cx(0, 1)
qc_decomposed.h(1)

print("Direct controlled operation:")
print(qc_controlled.draw())
print("\nElementary decomposition:")
print(qc_decomposed.draw())
print("\nEquivalent:", equal_up_to_global_phase(circuit_unitary(qc_controlled),
                                                circuit_unitary(qc_decomposed)))

# Subsection **2.5.3 Common algebraic identities**

In [ ]:
# Original circuit with redundant gates
qc = Circuit(1)
qc.rz(np.pi/4, 0)
qc.h(0)
qc.rz(np.pi/4, 0)
qc.h(0)

# Simplified version using H Rz H = Rx
qc_simplified = Circuit(1)
qc_simplified.rz(np.pi/4, 0)
qc_simplified.rx(np.pi/4, 0)

U1 = circuit_unitary(qc)
U2 = circuit_unitary(qc_simplified)

print(equal_up_to_global_phase(U1, U2))

# Subsection **2.6 Practical example: building and verifying circuits**

In [ ]:
theta = np.pi/3  # example angle

# Build the circuit
qc = Circuit(2)
qc.ry(theta, 0)
qc.cx(0, 1)

# Unitary from qcirclab
U_sim = circuit_unitary(qc)

# Analytical matrix in the book's ordering |00>, |01>, |10>, |11>
Ry = np.array([[np.cos(theta/2), -np.sin(theta/2)],
               [np.sin(theta/2),  np.cos(theta/2)]], dtype=complex)
I2 = np.eye(2, dtype=complex)

CNOT = np.array([[1,0,0,0],
                 [0,1,0,0],
                 [0,0,0,1],
                 [0,0,1,0]], dtype=complex)

U_theory = CNOT @ np.kron(Ry, I2)

print("Circuit:")
print(qc.draw())
print("\nEquivalent to analytical matrix:", np.allclose(U_sim, U_theory))
print("\nUnitary from simulator:")
print(U_sim)